# Feature Extraction for Grafana Logs

This notebook extracts comprehensive features from preprocessed Grafana logs for clustering analysis.

## Feature Categories:
1. **Numerical Features**: Metric values, timestamps, statistical aggregations
2. **Temporal Features**: Time-based patterns (hour, day, trends)
3. **Categorical Features**: Encoded dashboard, panel, service information
4. **Text Features**: TF-IDF and embeddings from query strings and panel titles
5. **Statistical Features**: Rolling statistics, anomaly scores
6. **Query Complexity Features**: Extracted from Prometheus queries

## Approach:
- Combine multiple feature types for robust representation
- Handle high-dimensional categorical data with encoding strategies
- Create time-series features for temporal patterns
- Normalize and scale features appropriately

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed data
df = pd.read_pickle('preprocessed_grafana_logs.pkl')

print(f"Loaded dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Numerical Features

In [ ]:
# Select numerical features
numerical_features = ['value', 'hour', 'day_of_week', 'minute', 'query_complexity', 'time_window_value']

# Create additional numerical features
df['value_log'] = np.log1p(df['value'].abs())  # Log-transformed value
df['value_squared'] = df['value'] ** 2
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_business_hours'] = df['hour'].between(9, 17).astype(int)

# Add to numerical features list
numerical_features.extend(['value_log', 'value_squared', 'is_weekend', 'is_business_hours'])

print("Numerical features:")
print(numerical_features)
print(f"\nNumerical features statistics:")
print(df[numerical_features].describe())

## 3. Statistical Features (Rolling and Aggregations)

In [ ]:
# Sort by timestamp for time-series features
df = df.sort_values('timestamp').reset_index(drop=True)

# Group by service and panel for service-specific features
df['service_panel'] = df['service'] + '_' + df['panel_title']

# Calculate rolling statistics per service-panel group
def add_rolling_features(group, windows=[10, 50, 100]):
    """
    Add rolling statistics for a group.
    """
    for window in windows:
        if len(group) >= window:
            group[f'rolling_mean_{window}'] = group['value'].rolling(window=window, min_periods=1).mean()
            group[f'rolling_std_{window}'] = group['value'].rolling(window=window, min_periods=1).std()
            group[f'rolling_min_{window}'] = group['value'].rolling(window=window, min_periods=1).min()
            group[f'rolling_max_{window}'] = group['value'].rolling(window=window, min_periods=1).max()
        else:
            group[f'rolling_mean_{window}'] = group['value'].mean()
            group[f'rolling_std_{window}'] = group['value'].std()
            group[f'rolling_min_{window}'] = group['value'].min()
            group[f'rolling_max_{window}'] = group['value'].max()
    
    return group

print("Computing rolling statistics...")
df = df.groupby('service_panel', group_keys=False).apply(add_rolling_features)

# Add rate of change features
df['value_diff'] = df.groupby('service_panel')['value'].diff()
df['value_pct_change'] = df.groupby('service_panel')['value'].pct_change()

# Fill NaN values from rolling calculations
rolling_cols = [col for col in df.columns if 'rolling' in col or 'diff' in col or 'pct_change' in col]
df[rolling_cols] = df[rolling_cols].fillna(0)

print("Rolling features added successfully!")
print(f"Rolling feature columns: {rolling_cols}")

## 4. Anomaly Score Features

In [ ]:
# Calculate z-score per service-panel group
def calculate_zscore(group):
    mean = group['value'].mean()
    std = group['value'].std()
    if std > 0:
        group['value_zscore'] = (group['value'] - mean) / std
    else:
        group['value_zscore'] = 0
    return group

df = df.groupby('service_panel', group_keys=False).apply(calculate_zscore)

# Anomaly flag based on z-score
df['is_anomaly_zscore'] = (df['value_zscore'].abs() > 3).astype(int)

# IQR-based anomaly detection
def calculate_iqr_anomaly(group):
    Q1 = group['value'].quantile(0.25)
    Q3 = group['value'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    group['is_anomaly_iqr'] = ((group['value'] < lower_bound) | (group['value'] > upper_bound)).astype(int)
    return group

df = df.groupby('service_panel', group_keys=False).apply(calculate_iqr_anomaly)

print("Anomaly features created!")
print(f"Z-score anomalies: {df['is_anomaly_zscore'].sum()}")
print(f"IQR anomalies: {df['is_anomaly_iqr'].sum()}")

## 5. Categorical Feature Encoding

In [ ]:
# Label encoding for categorical variables
categorical_columns = ['dashboard', 'panel_title', 'panel_type', 'datasource', 'service', 'environment']

label_encoders = {}
for col in categorical_columns:
    if col in df.columns:
        le = LabelEncoder()
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

print("Label encoding complete!")
print("\nEncoded column mappings:")
for col, le in label_encoders.items():
    print(f"  {col}: {len(le.classes_)} unique values")

# One-hot encoding for low-cardinality categorical variables
low_cardinality_cols = ['panel_type', 'datasource', 'environment']
df_encoded = pd.get_dummies(df, columns=low_cardinality_cols, prefix=low_cardinality_cols)

print(f"\nDataset shape after encoding: {df_encoded.shape}")

## 6. Text Features (TF-IDF)

In [ ]:
# TF-IDF on panel titles
print("Extracting TF-IDF features from panel titles...")
tfidf_panel = TfidfVectorizer(max_features=20, stop_words='english', ngram_range=(1, 2))
panel_tfidf = tfidf_panel.fit_transform(df_encoded['panel_title'].astype(str))
panel_tfidf_df = pd.DataFrame(
    panel_tfidf.toarray(),
    columns=[f'panel_tfidf_{i}' for i in range(panel_tfidf.shape[1])]
)

print(f"Panel TF-IDF features: {panel_tfidf_df.shape[1]}")
print(f"Top features: {tfidf_panel.get_feature_names_out()[:10]}")

# TF-IDF on Prometheus queries
print("\nExtracting TF-IDF features from queries...")
tfidf_query = TfidfVectorizer(max_features=30, ngram_range=(1, 2))
query_tfidf = tfidf_query.fit_transform(df_encoded['query'].fillna('').astype(str))
query_tfidf_df = pd.DataFrame(
    query_tfidf.toarray(),
    columns=[f'query_tfidf_{i}' for i in range(query_tfidf.shape[1])]
)

print(f"Query TF-IDF features: {query_tfidf_df.shape[1]}")
print(f"Top features: {tfidf_query.get_feature_names_out()[:10]}")

# Concatenate TF-IDF features
df_encoded = pd.concat([df_encoded.reset_index(drop=True), panel_tfidf_df, query_tfidf_df], axis=1)

print(f"\nDataset shape after TF-IDF: {df_encoded.shape}")

## 7. Query Feature Engineering

In [ ]:
# Boolean query features
query_boolean_features = [
    'has_rate', 'has_histogram_quantile', 'has_sum', 'has_count',
    'has_avg', 'has_max', 'has_min', 'is_cpu_metric', 'is_memory_metric',
    'is_network_metric', 'is_http_metric', 'is_database_metric', 'is_jvm_metric'
]

# Ensure boolean features are integers
for col in query_boolean_features:
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].fillna(0).astype(int)

print("Query boolean features:")
print(df_encoded[query_boolean_features].sum())

## 8. Feature Selection and Final Feature Set

In [ ]:
# Compile all feature columns for clustering
feature_columns = []

# Numerical features
feature_columns.extend(numerical_features)

# Rolling features
feature_columns.extend(rolling_cols)

# Anomaly features
feature_columns.extend(['value_zscore', 'is_anomaly_zscore', 'is_anomaly_iqr'])

# Encoded categorical features
feature_columns.extend([f'{col}_encoded' for col in categorical_columns if f'{col}_encoded' in df_encoded.columns])

# One-hot encoded features
onehot_cols = [col for col in df_encoded.columns if any(prefix in col for prefix in low_cardinality_cols)]
feature_columns.extend(onehot_cols)

# TF-IDF features
tfidf_cols = [col for col in df_encoded.columns if 'tfidf' in col]
feature_columns.extend(tfidf_cols)

# Query boolean features
feature_columns.extend([col for col in query_boolean_features if col in df_encoded.columns])

# Remove duplicates and ensure all columns exist
feature_columns = list(set(feature_columns))
feature_columns = [col for col in feature_columns if col in df_encoded.columns]

print(f"Total feature columns: {len(feature_columns)}")
print(f"\nFeature categories:")
print(f"  - Numerical: {len(numerical_features)}")
print(f"  - Rolling: {len(rolling_cols)}")
print(f"  - Anomaly: 3")
print(f"  - Encoded categorical: {len([col for col in feature_columns if '_encoded' in col])}")
print(f"  - One-hot: {len(onehot_cols)}")
print(f"  - TF-IDF: {len(tfidf_cols)}")
print(f"  - Query boolean: {len([col for col in query_boolean_features if col in df_encoded.columns])}")

# Extract feature matrix
X = df_encoded[feature_columns].copy()

# Handle any remaining NaN values
X = X.fillna(0)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Missing values: {X.isnull().sum().sum()}")

## 9. Feature Scaling and Normalization

In [ ]:
# StandardScaler for features (important for distance-based clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns)

print("Feature scaling complete!")
print(f"Scaled feature matrix shape: {X_scaled_df.shape}")
print(f"\nScaled feature statistics:")
print(X_scaled_df.describe())

## 10. Dimensionality Reduction (PCA) for Visualization

In [ ]:
# Apply PCA for visualization
pca = PCA(n_components=min(50, X_scaled.shape[1]))
X_pca = pca.fit_transform(X_scaled)

# Explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(range(1, len(explained_variance) + 1), explained_variance, 'bo-')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].grid(True)

axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'ro-')
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Find number of components for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"\nNumber of components for 95% variance: {n_components_95}")
print(f"Total explained variance by first {n_components_95} components: {cumulative_variance[n_components_95-1]:.4f}")

## 11. Feature Importance Analysis

In [ ]:
# Calculate feature importance based on variance
feature_variance = X_scaled_df.var().sort_values(ascending=False)

# Plot top 20 features by variance
plt.figure(figsize=(12, 6))
feature_variance.head(20).plot(kind='barh')
plt.xlabel('Variance')
plt.title('Top 20 Features by Variance')
plt.tight_layout()
plt.show()

print("Top 20 features by variance:")
print(feature_variance.head(20))

## 12. Correlation Analysis

In [ ]:
# Calculate correlation matrix for top features
top_features = feature_variance.head(20).index
correlation_matrix = X_scaled_df[top_features].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap (Top 20 Features)')
plt.tight_layout()
plt.show()

# Identify highly correlated features (> 0.9)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("\nHighly correlated feature pairs (|r| > 0.9):")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print("\nNo highly correlated features found (|r| > 0.9)")

## 13. Save Feature Matrices

In [ ]:
# Save unscaled features
X.to_csv('features_unscaled.csv', index=False)
print("Unscaled features saved to: features_unscaled.csv")

# Save scaled features
X_scaled_df.to_csv('features_scaled.csv', index=False)
print("Scaled features saved to: features_scaled.csv")

# Save PCA-transformed features
X_pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
X_pca_df.to_csv('features_pca.csv', index=False)
print(f"PCA features saved to: features_pca.csv")

# Save feature names for reference
with open('feature_names.txt', 'w') as f:
    for col in feature_columns:
        f.write(f"{col}\n")
print("Feature names saved to: feature_names.txt")

# Save metadata for clusters
metadata_cols = ['timestamp', 'dashboard', 'panel_title', 'service', 'value']
metadata = df_encoded[metadata_cols].copy()
metadata.to_csv('metadata.csv', index=False)
print("Metadata saved to: metadata.csv")

# Save preprocessing objects
import pickle

with open('preprocessing_objects.pkl', 'wb') as f:
    pickle.dump({
        'scaler': scaler,
        'pca': pca,
        'label_encoders': label_encoders,
        'tfidf_panel': tfidf_panel,
        'tfidf_query': tfidf_query,
        'feature_columns': feature_columns
    }, f)
print("Preprocessing objects saved to: preprocessing_objects.pkl")

print("\n" + "="*80)
print("FEATURE EXTRACTION SUMMARY")
print("="*80)
print(f"Total samples: {X_scaled.shape[0]:,}")
print(f"Total features: {X_scaled.shape[1]}")
print(f"PCA components (95% variance): {n_components_95}")
print(f"Feature types:")
print(f"  - Numerical: {len(numerical_features)}")
print(f"  - Statistical: {len(rolling_cols)}")
print(f"  - Categorical (encoded): {len([col for col in feature_columns if '_encoded' in col])}")
print(f"  - Text (TF-IDF): {len(tfidf_cols)}")
print(f"  - Boolean: {len([col for col in query_boolean_features if col in df_encoded.columns])}")
print("="*80)